# 🌿 PlantX Detector – Swin Transformer Training Notebook
**Plant Disease Detector** · trains a Swin-style Transformer on leaf images and exports the files used by the Flask app.

**Outputs (copy into the `models/` folder of the web app):**
- `models/plantx_swin.weights.h5` – best trained weights
- `models/plantx_classes.json` – ordered class names

**Environment:** Python 3.8/3.9 and `pip install -r requirements.txt` (TensorFlow 2.7.0, tensorflow-addons 0.14.0).
The model code below is identical to `plantx_model.py`, which the backend imports, so training and serving always match.

In [ ]:
# Install the pinned dependencies (run once, from the project folder)
# !pip install -r requirements.txt

## 1 · Imports

In [ ]:
import os, json, random
from os import listdir

import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator

try:
    import tensorflow_addons as tfa      # provides AdamW for TF 2.7
except ImportError:
    tfa = None

print('TensorFlow', tf.__version__, '| TFA', getattr(tfa, '__version__', 'not installed'))

## 2 · Model definition (shared with the web backend)

In [ ]:
import cv2
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

### Hyper-parameters

In [ ]:
IMG_SIDE = 64                       # images are resized to IMG_SIDE x IMG_SIDE
INPUT_SHAPE = (IMG_SIDE, IMG_SIDE, 3)
PIXEL_SCALE = 255.0                 # pixel normalisation divisor
PATCH_SIZE = (4, 4)                 # size of each image patch
EMBED_DIM = 256                     # token embedding width
NUM_HEADS = 4                       # attention heads
MLP_UNITS = 256                     # hidden units in the transformer MLP
QKV_BIAS = True                     # bias in query/key/value projections
WINDOW = 2                          # attention window size
SHIFT = 2                           # shifted-window offset
DROP_RATE = 0.2                     # dropout / stochastic-depth rate

PATCHES_X = INPUT_SHAPE[0] // PATCH_SIZE[0]
PATCHES_Y = INPUT_SHAPE[1] // PATCH_SIZE[1]

DEFAULT_CLASSES = ["bacterial_blight", "curl_virus", "fussarium_wilt", "healthy"]

### Image preprocessing helper

In [ ]:
def prepare_image(bgr_image):
    """Resize a BGR (cv2) image and scale it to [0, 1]. Returns float32 HxWx3."""
    resized = cv2.resize(bgr_image, (IMG_SIDE, IMG_SIDE))
    return resized.astype("float32") / PIXEL_SCALE

### Window helpers

In [ ]:
def split_into_windows(feature_map, win):
    _, h, w, c = feature_map.shape
    rows, cols = h // win, w // win
    feature_map = tf.reshape(feature_map, (-1, rows, win, cols, win, c))
    feature_map = tf.transpose(feature_map, (0, 1, 3, 2, 4, 5))
    return tf.reshape(feature_map, (-1, win, win, c))


def merge_windows(windows, win, h, w, c):
    rows, cols = h // win, w // win
    merged = tf.reshape(windows, (-1, rows, cols, win, win, c))
    merged = tf.transpose(merged, (0, 1, 3, 2, 4, 5))
    return tf.reshape(merged, (-1, h, w, c))


class StochasticDepth(layers.Layer):
    """Drops whole residual branches - ONLY while training."""

    def __init__(self, rate=0.0, **kwargs):
        super().__init__(**kwargs)
        self.rate = rate

    def call(self, inputs, training=None):
        if not training or not self.rate:
            return inputs
        batch = tf.shape(inputs)[0]
        shape = (batch,) + (1,) * (inputs.shape.rank - 1)
        keep = 1.0 - self.rate
        mask = tf.floor(keep + tf.random.uniform(shape, dtype=inputs.dtype))
        return tf.math.divide(inputs, keep) * mask

### Attention and transformer block

In [ ]:
class PlantXWindowAttention(layers.Layer):
    def __init__(self, dim, window, heads, qkv_bias=True, drop=0.0, **kwargs):
        super().__init__(**kwargs)
        self.dim = dim
        self.window = window
        self.heads = heads
        self.scale = (dim // heads) ** -0.5
        self.qkv_dense = layers.Dense(dim * 3, use_bias=qkv_bias)
        self.attn_drop = layers.Dropout(drop)
        self.out_dense = layers.Dense(dim)

    def build(self, input_shape):
        table_rows = (2 * self.window[0] - 1) * (2 * self.window[1] - 1)
        self.rel_bias_table = self.add_weight(
            name="rel_bias_table",
            shape=(table_rows, self.heads),
            initializer=tf.initializers.Zeros(),
            trainable=True,
        )
        ys = np.arange(self.window[0])
        xs = np.arange(self.window[1])
        grid = np.stack(np.meshgrid(ys, xs, indexing="ij"))
        flat = grid.reshape(2, -1)
        rel = flat[:, :, None] - flat[:, None, :]
        rel = rel.transpose([1, 2, 0])
        rel[:, :, 0] += self.window[0] - 1
        rel[:, :, 1] += self.window[1] - 1
        rel[:, :, 0] *= 2 * self.window[1] - 1
        self.rel_index = tf.Variable(
            initial_value=tf.convert_to_tensor(rel.sum(-1)),
            trainable=False, name="rel_index",
        )

    def call(self, tokens, mask=None, training=None):
        _, n_tok, channels = tokens.shape
        head_dim = channels // self.heads
        qkv = self.qkv_dense(tokens)
        qkv = tf.reshape(qkv, (-1, n_tok, 3, self.heads, head_dim))
        qkv = tf.transpose(qkv, (2, 0, 3, 1, 4))
        q, k, v = qkv[0], qkv[1], qkv[2]
        scores = (q * self.scale) @ tf.transpose(k, (0, 1, 3, 2))

        win_area = self.window[0] * self.window[1]
        bias = tf.gather(self.rel_bias_table, tf.reshape(self.rel_index, (-1,)))
        bias = tf.reshape(bias, (win_area, win_area, -1))
        bias = tf.transpose(bias, (2, 0, 1))
        scores = scores + tf.expand_dims(bias, 0)

        if mask is not None:
            n_win = mask.get_shape()[0]
            mask_f = tf.cast(tf.expand_dims(tf.expand_dims(mask, 1), 0), tf.float32)
            scores = tf.reshape(scores, (-1, n_win, self.heads, n_tok, n_tok)) + mask_f
            scores = tf.reshape(scores, (-1, self.heads, n_tok, n_tok))
        weights = keras.activations.softmax(scores, axis=-1)
        weights = self.attn_drop(weights, training=training)

        out = weights @ v
        out = tf.transpose(out, (0, 2, 1, 3))
        out = tf.reshape(out, (-1, n_tok, channels))
        out = self.out_dense(out)
        return self.attn_drop(out, training=training)


class PlantXSwinBlock(layers.Layer):
    def __init__(self, dim, grid, heads, window=7, shift=0, mlp_units=1024,
                 qkv_bias=True, drop=0.0, **kwargs):
        super().__init__(**kwargs)
        self.dim = dim
        self.grid = grid
        self.heads = heads
        self.window = window
        self.shift = shift
        self.mlp_units = mlp_units

        self.norm_a = layers.LayerNormalization(epsilon=1e-5)
        self.attention = PlantXWindowAttention(
            dim, (self.window, self.window), heads, qkv_bias=qkv_bias, drop=drop
        )
        self.path_drop = StochasticDepth(drop)
        self.norm_b = layers.LayerNormalization(epsilon=1e-5)
        self.mlp_dense_1 = layers.Dense(mlp_units)
        self.mlp_act = layers.Activation(keras.activations.gelu)
        self.mlp_drop_1 = layers.Dropout(drop)
        self.mlp_dense_2 = layers.Dense(dim)
        self.mlp_drop_2 = layers.Dropout(drop)

        if min(self.grid) < self.window:
            self.shift = 0
            self.window = min(self.grid)

    def build(self, input_shape):
        if self.shift == 0:
            self.attn_mask = None
            return
        h, w = self.grid
        h_cuts = (slice(0, -self.window), slice(-self.window, -self.shift),
                  slice(-self.shift, None))
        w_cuts = h_cuts
        canvas = np.zeros((1, h, w, 1))
        region = 0
        for hs in h_cuts:
            for ws in w_cuts:
                canvas[:, hs, ws, :] = region
                region += 1
        canvas = tf.convert_to_tensor(canvas)
        win_ids = split_into_windows(canvas, self.window)
        win_ids = tf.reshape(win_ids, (-1, self.window * self.window))
        diff = tf.expand_dims(win_ids, 1) - tf.expand_dims(win_ids, 2)
        diff = tf.where(diff != 0, -100.0, diff)
        diff = tf.where(diff == 0, 0.0, diff)
        self.attn_mask = tf.Variable(initial_value=diff, trainable=False, name="attn_mask")

    def _mlp(self, x, training):
        x = self.mlp_dense_1(x)
        x = self.mlp_act(x)
        x = self.mlp_drop_1(x, training=training)
        x = self.mlp_dense_2(x)
        return self.mlp_drop_2(x, training=training)

    def call(self, x, training=None):
        h, w = self.grid
        _, _, channels = x.shape
        shortcut = x
        x = self.norm_a(x)
        x = tf.reshape(x, (-1, h, w, channels))
        if self.shift > 0:
            x = tf.roll(x, shift=[-self.shift, -self.shift], axis=[1, 2])

        wins = split_into_windows(x, self.window)
        wins = tf.reshape(wins, (-1, self.window * self.window, channels))
        wins = self.attention(wins, mask=self.attn_mask, training=training)
        wins = tf.reshape(wins, (-1, self.window, self.window, channels))
        x = merge_windows(wins, self.window, h, w, channels)

        if self.shift > 0:
            x = tf.roll(x, shift=[self.shift, self.shift], axis=[1, 2])

        x = tf.reshape(x, (-1, h * w, channels))
        x = shortcut + self.path_drop(x, training=training)
        shortcut = x
        x = self.norm_b(x)
        x = self._mlp(x, training)
        return shortcut + self.path_drop(x, training=training)

### Patch layers

In [ ]:
class PatchSlicer(layers.Layer):
    def __init__(self, patch_size, **kwargs):
        super().__init__(**kwargs)
        self.px = patch_size[0]
        self.py = patch_size[0]

    def call(self, images):
        batch = tf.shape(images)[0]
        patches = tf.image.extract_patches(
            images=images,
            sizes=(1, self.px, self.py, 1),
            strides=(1, self.px, self.py, 1),
            rates=(1, 1, 1, 1),
            padding="VALID",
        )
        depth = patches.shape[-1]
        count = patches.shape[1]
        return tf.reshape(patches, (batch, count * count, depth))


class PatchProjector(layers.Layer):
    def __init__(self, n_patches, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.n_patches = n_patches
        self.linear = layers.Dense(embed_dim)
        self.position = layers.Embedding(input_dim=n_patches, output_dim=embed_dim)

    def call(self, patches):
        idx = tf.range(start=0, limit=self.n_patches, delta=1)
        return self.linear(patches) + self.position(idx)


class PatchDownsample(layers.Layer):
    def __init__(self, grid, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.grid = grid
        self.embed_dim = embed_dim
        self.reduce = layers.Dense(2 * embed_dim, use_bias=False)

    def call(self, x):
        h, w = self.grid
        _, _, c = x.get_shape().as_list()
        x = tf.reshape(x, (-1, h, w, c))
        merged = tf.concat(
            (x[:, 0::2, 0::2, :], x[:, 1::2, 0::2, :],
             x[:, 0::2, 1::2, :], x[:, 1::2, 1::2, :]), axis=-1)
        merged = tf.reshape(merged, (-1, (h // 2) * (w // 2), 4 * c))
        return self.reduce(merged)

### Model factory

In [ ]:
def build_plantx_model(n_classes):
    """Builds the PlantX Swin-style classifier (uncompiled)."""
    inp = layers.Input(INPUT_SHAPE)
    x = inp
    x = PatchSlicer(PATCH_SIZE)(x)
    x = PatchProjector(PATCHES_X * PATCHES_Y, EMBED_DIM)(x)
    x = PlantXSwinBlock(
        dim=EMBED_DIM, grid=(PATCHES_X, PATCHES_Y), heads=NUM_HEADS,
        window=WINDOW, shift=0, mlp_units=MLP_UNITS,
        qkv_bias=QKV_BIAS, drop=DROP_RATE)(x)
    x = PlantXSwinBlock(
        dim=EMBED_DIM, grid=(PATCHES_X, PATCHES_Y), heads=NUM_HEADS,
        window=WINDOW, shift=SHIFT, mlp_units=MLP_UNITS,
        qkv_bias=QKV_BIAS, drop=DROP_RATE)(x)
    x = PatchDownsample((PATCHES_X, PATCHES_Y), embed_dim=EMBED_DIM)(x)
    x = layers.GlobalAveragePooling1D()(x)
    out = layers.Dense(n_classes, activation="softmax")(x)
    return keras.Model(inp, out, name="plantx_swin")

## 3 · Training configuration

In [ ]:
# ---- dataset (folder structure: <DATA_ROOT>/train/<class_name>/*.jpg) ----
DATA_ROOT   = './dataset'          # change to your dataset path
# Google Colab users: uncomment the next three lines
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_ROOT = '/content/drive/MyDrive/Cotton Plant Disease Project/Cotton Crop Disease Dataset'

TRAIN_FOLDER   = os.path.join(DATA_ROOT, 'train')
MAX_PER_CLASS  = 5816              # cap on images read per class
VALID_EXT      = ('.jpg', '.jpeg', '.png')

# ---- optimisation ----
LEARNING_RATE   = 1e-2
BATCH           = 32
EPOCHS          = 100
WEIGHT_DECAY    = 0.01
LABEL_SMOOTHING = 0.1
TEST_FRACTION   = 0.2
SEED            = 42

# ---- output files read by the Flask app ----
OUT_DIR      = 'models'
WEIGHTS_FILE = os.path.join(OUT_DIR, 'plantx_swin.weights.h5')
CLASSES_FILE = os.path.join(OUT_DIR, 'plantx_classes.json')
os.makedirs(OUT_DIR, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

## 4 · Load the images

In [ ]:
def load_leaf(path):
    """Read one image file and return the preprocessed array, or None if unreadable."""
    bgr = cv2.imread(path)
    if bgr is None:
        return None
    return pm_prepare(bgr)

pm_prepare = prepare_image   # alias to the helper defined above

leaf_arrays, leaf_labels = [], []
print('[INFO] Loading images ...')
for disease_name in sorted(listdir(TRAIN_FOLDER)):
    class_path = os.path.join(TRAIN_FOLDER, disease_name)
    if not os.path.isdir(class_path):
        continue
    print(f'[INFO] Processing {disease_name} ...')
    for file_name in listdir(class_path)[:MAX_PER_CLASS]:
        if not file_name.lower().endswith(VALID_EXT):
            continue
        arr = load_leaf(os.path.join(class_path, file_name))
        if arr is not None:
            leaf_arrays.append(arr)
            leaf_labels.append(disease_name)

X_all = np.array(leaf_arrays, dtype='float32')
print(f'[INFO] Loaded {len(X_all)} images -> array shape {X_all.shape}')

In [ ]:
binarizer = LabelBinarizer()
Y_all = binarizer.fit_transform(leaf_labels)
CLASS_NAMES = list(binarizer.classes_)
N_CLASSES = len(CLASS_NAMES)

with open(CLASSES_FILE, 'w', encoding='utf-8') as fh:
    json.dump(CLASS_NAMES, fh, indent=2)
print('Classes:', CLASS_NAMES, '->', CLASSES_FILE)

In [ ]:
# Quick look at a few samples
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, idx in zip(axes.ravel(), np.random.choice(len(X_all), 8, replace=False)):
    ax.imshow(X_all[idx][..., ::-1])          # BGR -> RGB for display
    ax.set_title(leaf_labels[idx]); ax.axis('off')
plt.tight_layout(); plt.show()

## 5 · Split + augmentation

In [ ]:
X_train, X_valid, Y_train, Y_valid = train_test_split(
    X_all, Y_all, test_size=TEST_FRACTION, random_state=SEED, stratify=Y_all)
print('train:', X_train.shape, Y_train.shape)
print('valid:', X_valid.shape, Y_valid.shape)

augmenter = ImageDataGenerator(
    rotation_range=25, width_shift_range=0.1, height_shift_range=0.1,
    shear_range=0.2, zoom_range=0.2, horizontal_flip=True, fill_mode='nearest')

## 6 · Build and compile

In [ ]:
tf.compat.v1.logging.set_verbosity(tf.compat.v1.logging.ERROR)

net = build_plantx_model(N_CLASSES)

# AdamW (decoupled weight decay) comes from tensorflow-addons on TF 2.7
if tfa is not None:
    optimiser = tfa.optimizers.AdamW(weight_decay=WEIGHT_DECAY, learning_rate=LEARNING_RATE)
else:
    print('tensorflow-addons missing -> falling back to plain Adam (no weight decay)')
    optimiser = keras.optimizers.Adam(learning_rate=LEARNING_RATE)

net.compile(
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
    optimizer=optimiser,
    metrics=[keras.metrics.CategoricalAccuracy(name='accuracy')])
net.summary()

## 7 · Train

In [ ]:
training_callbacks = [
    ReduceLROnPlateau(monitor='val_loss', factor=0.1, patience=10, min_lr=1e-10, verbose=1),
    ModelCheckpoint(WEIGHTS_FILE, monitor='val_accuracy', mode='max',
                    save_best_only=True, save_weights_only=True, verbose=1),
]

print('[INFO] Training network ...')
run = net.fit(
    augmenter.flow(X_train, Y_train, batch_size=BATCH),
    validation_data=(X_valid, Y_valid),
    steps_per_epoch=int(np.ceil(len(X_train) / float(BATCH))),
    epochs=EPOCHS, verbose=1, callbacks=training_callbacks)

## 8 · Training curves

In [ ]:
hist = run.history
fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(14, 5))

ax_loss.plot(hist['loss'], label='training');       ax_loss.plot(hist['val_loss'], label='validation')
ax_loss.set(title='Loss', xlabel='epoch', ylabel='loss');       ax_loss.legend(); ax_loss.grid(alpha=.3)

ax_acc.plot(hist['accuracy'], label='training');    ax_acc.plot(hist['val_accuracy'], label='validation')
ax_acc.set(title='Accuracy', xlabel='epoch', ylabel='accuracy'); ax_acc.legend(); ax_acc.grid(alpha=.3)
plt.tight_layout(); plt.show()

## 9 · Evaluate the best checkpoint

In [ ]:
net.load_weights(WEIGHTS_FILE)                     # best epoch by validation accuracy
val_loss, val_acc = net.evaluate(X_valid, Y_valid, verbose=0)
print(f'[INFO] Validation accuracy: {val_acc * 100:.2f}%')

y_true = np.argmax(Y_valid, axis=1)
y_pred = np.argmax(net.predict(X_valid), axis=1)
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5)); plt.imshow(cm, cmap='Greens'); plt.colorbar()
plt.xticks(range(N_CLASSES), CLASS_NAMES, rotation=35, ha='right'); plt.yticks(range(N_CLASSES), CLASS_NAMES)
for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        plt.text(j, i, cm[i, j], ha='center', va='center')
plt.xlabel('predicted'); plt.ylabel('actual'); plt.title('Confusion matrix'); plt.tight_layout(); plt.show()

## 10 · Optional: ensemble of several trained models

In [ ]:
def ensemble_predict(model_list, data, weights=None):
    """Weighted average of the softmax outputs of several models."""
    if weights is None:
        weights = [1 / len(model_list)] * len(model_list)
    blended = np.zeros((data.shape[0], N_CLASSES))
    for m, w in zip(model_list, weights):
        blended += m.predict(data) * w
    return blended

def ensemble_accuracy(prob_list, truth_onehot, weights=None):
    if weights is None:
        weights = [1 / len(prob_list)] * len(prob_list)
    blended = sum(p * w for p, w in zip(prob_list, weights))
    return accuracy_score(np.argmax(truth_onehot, axis=1), np.argmax(blended, axis=1))

## 11 · Predict a single leaf

In [ ]:
def predict_leaf(image_path):
    bgr = cv2.imread(image_path)
    if bgr is None:
        raise FileNotFoundError(image_path)
    probs = net.predict(np.expand_dims(prepare_image(bgr), 0), verbose=0)[0]
    best = int(np.argmax(probs))
    plt.imshow(bgr[..., ::-1]); plt.axis('off')
    plt.title(f'{CLASS_NAMES[best]}  ({probs[best] * 100:.1f}%)')
    plt.show()
    return dict(zip(CLASS_NAMES, np.round(probs * 100, 2)))

# predict_leaf('./dataset/val/healthy/example.jpg')

## 12 · Export for the web app

In [ ]:
# Weights were already written by ModelCheckpoint; re-save the final (best) ones explicitly.
net.save_weights(WEIGHTS_FILE)
print('Saved:', WEIGHTS_FILE)
print('Saved:', CLASSES_FILE)
print('\nNext: run  python app.py  and open http://127.0.0.1:5000')